## Question 4 (Bonus) — Validity of Regression on an MCMC Sample

OLS regression implicitly assumes **independent** observations. An MCMC chain produces **correlated** samples, which does not bias the ZV estimator (consistency follows from the ergodic TCL) but **underestimates the standard error** of the OLS coefficients. In other words, $\hat{\beta}$ is correct, but $\text{Var}(\hat{\beta})$ is poorly estimated if the dependence is ignored.

### Strategies to mitigate the problem
1. **Thinning**: retain only one sample every $k$ steps, where $k$ is chosen so that autocorrelation is negligible. Simple but costly in terms of effective sample size.
2. **Batch means**: divide the time series into $B$ blocks of size $m$, and use the block means as i.i.d. pseudo-observations. A good compromise between independence and sample size.
3. **HAC-type standard error** (Newey-West): estimate $\text{Var}(\hat{\beta})$ in a way that is robust to serial correlation without modifying the time series.

In [3]:
# ─────────────────────────────────────────────────────────────
# Bonus — Comparison of raw ZV-OLS vs. subsampled vs. block
# ─────────────────────────────────────────────────────────────

def zv_thinned(chain, Z, W, f_idx=0, thin=5):
    """ ZV-OLS on a thinned chain. """
    ch_ = chain[::thin]; W_ = W[::thin]
    f_  = ch_[:, f_idx]
    mu, _ = zv_estimator_ols(f_, W_)
    return mu


def zv_batch_means(chain, Z, W, f_idx=0, block_size=50):
    """-OLS on block means."""
    N  = len(chain)
    B  = N // block_size
    f_blocks = np.array([chain[b*block_size:(b+1)*block_size, f_idx].mean() for b in range(B)])
    W_blocks = np.array([W[b*block_size:(b+1)*block_size].mean(axis=0)       for b in range(B)])
    mu, _ = zv_estimator_ols(f_blocks, W_blocks)
    return mu


# Illustration using simulated data, parameter ω₁
f_test = chain_q3s[:, 0]
mu_raw    = zv_estimator_ols(f_test, W2_q3s)[0]
mu_thin5  = zv_thinned(chain_q3s, Z_q3s, W2_q3s, f_idx=0, thin=5)
mu_thin10 = zv_thinned(chain_q3s, Z_q3s, W2_q3s, f_idx=0, thin=10)
mu_block  = zv_batch_means(chain_q3s, Z_q3s, W2_q3s, f_idx=0, block_size=100)

print("Bonus — Estimators for ω₁ (simulated data, degree 2)")
print(f'  True value        : {omega_true[0]}')
print(f'  Raw MC             : {f_test.mean():.5f}')
print(f'  Raw ZV-OLS         : {mu_raw:.5f}')
print(f'  ZV-OLS thin=5       : {mu_thin5:.5f}')
print(f'  ZV-OLS thin=10      : {mu_thin10:.5f}')
print(f'  ZV-OLS block (m=100) : {mu_block:.5f}')

NameError: name 'chain_q3s' is not defined

In [ ]:
# Comparaison sur données réelles
if HAS_YFINANCE:
    for fi, nm in enumerate(['w1', 'w2', 'w3']):
        f_  = chain_q3r[:, fi]
        mu0 = zv_estimator_ols(f_, W2_q3r)[0]
        mu5 = zv_thinned(chain_q3r, Z_q3r, W2_q3r, f_idx=fi, thin=5)
        mub = zv_batch_means(chain_q3r, Z_q3r, W2_q3r, f_idx=fi, block_size=100)
        print(f'{nm:4s}  MC={f_.mean():.6f}  ZV-brut={mu0:.6f}  ZV-thin5={mu5:.6f}  ZV-bloc={mub:.6f}')